# 🚀 LLM in Python - From Simple to Complex

This notebook walks through practical Groq API examples, progressively building in complexity.

| # | Example | Difficulty |
|---|---------|------------|
| 1 | Setup & Basic Chat | ⭐ |
| 2 | System Prompts & Personas | ⭐⭐ |
| 3 | Streaming Responses | ⭐⭐ |
| 6 | Sentiment Analysis Pipeline | ⭐⭐⭐ |

**> Get your free API key at:** https://console.groq.com/keys

## 🔧 Setup — Install & Configure

In [1]:
!pip install groq faiss-cpu sentence-transformers ipywidgets -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.7/139.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 69.1 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

# Option A: Store your key in Colab Secrets (recommended)
# Go to the key icon in the left sidebar and add GROQ_API_KEY
try:
    GROQ_API_KEY = userdata.get('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')
    print('Loaded API key from Colab Secrets')
except Exception:
    GROQ_API_KEY = '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'  # Replace with your key
    print('Using hardcoded key - prefer Colab Secrets for security')

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

Using hardcoded key - prefer Colab Secrets for security


In [3]:
from groq import Groq

client = Groq(api_key=GROQ_API_KEY)

MODELS = {
    'fast':     'llama-3.1-8b-instant',
    'balanced': 'llama-3.3-70b-versatile',
    'code':     'qwen-2.5-coder-32b',
}

print('Groq client ready!')
print('Models:', list(MODELS.keys()))

Groq client ready!
Models: ['fast', 'balanced', 'code']


---
## Example 1 - Basic Chat Completion

In [4]:
response = client.chat.completions.create(
    model=MODELS['fast'],
    messages=[
        {"role": "user", "content": "What is Groq and why is it so fast?"}
    ]
)

print(response.choices[0].message.content)
print(f"\nTokens used: {response.usage.total_tokens}")
print(f"Speed: {response.usage.completion_tokens / response.usage.completion_time:.0f} tokens/sec")

Groq is a technology company that specializes in creating high-performance computing chips for artificial intelligence (AI) and machine learning (ML) applications. Their focus is on developing custom Application-Specific Integrated Circuits (ASICs) designed to optimize performance and energy efficiency for AI workloads.

Groq's technology is based on their proprietary software and hardware stacks, which they claim provide up to 100 times better performance and 80 times better power efficiency compared to traditional data center architectures. Their custom ASICs are designed to process large amounts of data in parallel, which enables them to achieve high speeds and throughput.

Groq's secret to speed is attributed to several key factors:

1. **Parallel processing**: Their ASICs are designed to process multiple data streams simultaneously, which increases processing speed and reduces latency. By exploiting parallelism in AI workloads, Groq's chips can process large batches of data in a r

---
## Example 2 - System Prompts & Personas

In [5]:
def chat_with_persona(persona_prompt, user_message, model=None):
    model = model or MODELS['fast']
    response = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": persona_prompt},
            {"role": "user",   "content": user_message}
        ],
        temperature=0.7,
        max_tokens=512
    )
    return response.choices[0].message.content


socratic = """
You are a Socratic teacher. Never give direct answers.
Guide the student with thoughtful questions only. Keep responses under 80 words.
"""
print("Socratic Teacher:")
print(chat_with_persona(socratic, "Why does the sky appear blue?"))

print("\n" + "-"*60 + "\n")

pirate_chef = """
You are a salty old pirate who is also a Michelin-star chef.
Speak in pirate dialect but give genuine, expert cooking advice.
"""
print("Pirate Chef:")
print(chat_with_persona(pirate_chef, "How do I make the perfect risotto?"))

Socratic Teacher:
What do you think happens when sunlight enters Earth's atmosphere? Does it change in any way before reaching our eyes? Can you think of a reason why we might see different colors under certain conditions, like during sunrise or sunset?

------------------------------------------------------------

Pirate Chef:
Arrr, ye landlubbers want ta know the secret to the perfect risotto, eh? Alright then, listen close and I'll share me treasure with ye. But first, grab yerself a pint o' good grog and sit yerself down, 'cause this be a long tale.

First, ye need ta choose the right rice, me hearty. I be talkin' about Arborio, the king o' risotto rice. It's got a special starch that absorbs liquid like a sponge and gives ye a creamy texture. Don't even think about usin' other types o' rice, or ye'll be walkin' the plank!

Now, let's get to the cookin' part. In a large pot, heat up some good, extra-virgin olive oil (not that watered-down bilge, mind ye). Then, add in some chopped 

---
## Example 3 - Sentiment Analysis Pipeline

Process a batch of customer reviews, classify each with fine-grained sentiment, extract topics, and produce a summary report.

In [18]:
import json
import pandas as pd

reviews = [
    "Absolutely love this product! Fast shipping and great quality. Will buy again.",
    "Terrible experience. Item arrived broken and customer service was unhelpful.",
    "It's okay. Does what it's supposed to do but nothing special.",
    "Amazing quality for the price! Blew my expectations out of the water.",
    "Returned it after 2 days. Poor build quality and the size was wrong.",
    "Decent product but delivery took 3 weeks longer than expected.",
    "Five stars! The packaging was beautiful and the item works perfectly.",
    "Not bad, not great. Instructions were confusing but it works fine now.",
]

def analyze_sentiment(review):
    response = client.chat.completions.create(
        model=MODELS["fast"],
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a sentiment analysis engine.\n\n"
                    "Return ONLY a JSON object with these exact keys:\n"

                    '  "sentiment": one of ["very_positive", "positive", "neutral", "negative", "very_negative"]\n'
                    '  "score": float from -1.0 to 1.0\n'
                    '  "topics": list of mentioned topics\n'
                    '  "emotion": dominant emotion\n'
                    '  "summary": a single sentence summary of the review\n\n'

                    "Use these sentiment rules strictly:\n"
                    "- very_positive: strongly enthusiastic, highly satisfied, glowing praise\n"
                    "- positive: generally satisfied, good overall, minor praise without extreme enthusiasm\n"
                    "- neutral: mixed, balanced, or plainly factual with little emotion\n"
                    "- negative: generally dissatisfied, some criticism, but not strongly emotional or catastrophic\n"
                    "- very_negative: strongly upset, angry, severe disappointment, or major failure\n\n"
                    "Use the full label range. Do not overuse very_positive or very_negative.\n"
                    "If a review contains both pros and cons, prefer neutral, positive, or negative instead of extreme labels.\n"
                    "Make the score align with the sentiment:\n"
                    "- very_positive: 0.75 to 1.0\n"
                    "- positive: 0.25 to 0.74\n"
                    "- neutral: -0.24 to 0.24\n"
                    "- negative: -0.74 to -0.25\n"
                    "- very_negative: -1.0 to -0.75"
                )
            },
            {"role": "user", "content": review}
        ],
        response_format={"type": "json_object"},
        temperature=0
    )

    result = json.loads(response.choices[0].message.content)
    result["review"] = review
    return result

results = [analyze_sentiment(r) for r in reviews]

EMOJI = {
    "very_positive": "😍",
    "positive": "😊",
    "neutral": "😐",
    "negative": "😞",
    "very_negative": "😡",
}

df = pd.DataFrame(results)

df["sentiment"] = df["sentiment"].astype(str).str.strip().str.lower()
df["emotion"] = df["emotion"].astype(str).str.strip().str.lower()
df["emoji"] = df["sentiment"].map(EMOJI)
df["topics"] = df["topics"].apply(lambda x: ", ".join(x) if isinstance(x, list) else str(x))

df = df[["emoji", "review", "sentiment", "score", "topics", "emotion", "summary"]]

df

,emoji,review,sentiment,score,topics,emotion,summary
0,😍,Absolutely love this product! Fast shipping an...,very_positive,0.95,"product, shipping, quality",enthusiasm,The reviewer is extremely satisfied with the p...
1,😡,Terrible experience. Item arrived broken and c...,very_negative,-0.92,"broken item, customer service",anger,The customer had a terrible experience with a ...
2,😐,It's okay. Does what it's supposed to do but n...,neutral,0.00,performance,neutral,It's okay.
3,😍,Amazing quality for the price! Blew my expecta...,very_positive,0.95,"quality, price",enthusiasm,The product exceeded expectations in terms of ...
4,😞,Returned it after 2 days. Poor build quality a...,negative,-0.62,"build quality, size",disappointment,The product has poor build quality and the wro...
5,😞,Decent product but delivery took 3 weeks longe...,negative,-0.55,"product, delivery",frustration,"The product was decent, but the delivery was d..."
6,😍,Five stars! The packaging was beautiful and th...,very_positive,0.95,"packaging, item",enthusiasm,The item has beautiful packaging and works per...
7,😐,"Not bad, not great. Instructions were confusin...",neutral,0.00,"instructions, software",neutral,"The software works fine, but the instructions ..."


---
## Quick Reference

| Concept | Key param / method |
|---|---|
| Pick a model | `model=` |
| Limit output | `max_tokens=` |
| Use tools | `tools=[...]`, `tool_choice="auto"` |
| OpenAI compat | `base_url="https://api.groq.com/openai/v1"` |

**Useful links:**
- [Groq Docs](https://console.groq.com/docs)
- [API Keys](https://console.groq.com/keys)
- [Model List](https://console.groq.com/docs/models)
- [Discord Community](https://discord.gg/groq)